![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
***
**Learning objectives**
- Understand gradient boosting conceptually (additive trees, residual fitting)
- Train XGBoost and LightGBM with clinical-appropriate hyperparameters
- Use `Optuna` for efficient Bayesian hyperparameter optimisation
- Handle categorical features natively in LightGBM
- Interpret early stopping and learning curves to prevent overfitting

**Estimated time:** 2.5 hours | **Level:** Intermediate → Advanced | **Libraries:** `xgboost`, `lightgbm`, `optuna`, `sklearn`


## 1. Setup & Dataset

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              brier_score_loss, roc_curve)
from sklearn.linear_model import LogisticRegression
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04",exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

try:
    import xgboost as xgb
    print(f"XGBoost {xgb.__version__}")
except ImportError:
    print("pip install xgboost")
try:
    import lightgbm as lgb
    print(f"LightGBM {lgb.__version__}")
except ImportError:
    print("pip install lightgbm")
try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f"Optuna {optuna.__version__}")
except ImportError:
    print("pip install optuna")

# ── Dataset ───────────────────────────────────────────────────────────────────
np.random.seed(42); N=3000
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,4)); np.random.seed(99)
age=np.random.normal(62,15,N).clip(18,92).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
payer=np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type=np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days=np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5+0.02*(age-62)/15),N)
hypertension=np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity=np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression=np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb=diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose=np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine=np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c=np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp=np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi=np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
       +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
       +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit),N)
df=pd.DataFrame({
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,
    "ckd":ckd,"obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,
    "readmitted":readmitted,
})
df["los_gt7"]=(df.los_days>7).astype(int)
NUMERIC=["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY=["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC=["sex","payer","admit_type"]
FEATURES=NUMERIC+BINARY+CATEGORIC; TARGET="readmitted"

pre=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),NUMERIC),
    ("bin",SimpleImputer(strategy="most_frequent"),BINARY),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),CATEGORIC),
])
X=df[FEATURES]; y=df[TARGET]
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
X_tr2,X_val,y_tr2,y_val=train_test_split(X_tr,y_tr,test_size=0.15,stratify=y_tr,random_state=42)
pre.fit(X_tr2)
Xtr=pre.transform(X_tr2); Xval=pre.transform(X_val); Xte=pre.transform(X_te)
ohe_nm=pre.named_transformers_["cat"]["ohe"].get_feature_names_out(CATEGORIC)
feat_nm=NUMERIC+BINARY+list(ohe_nm)
scale_pos = (y_tr2==0).sum()/(y_tr2==1).sum()
print(f"Train:{Xtr.shape} | Val:{Xval.shape} | Test:{Xte.shape}")
print(f"scale_pos_weight for XGBoost: {scale_pos:.2f}")


## 2. Gradient Boosting — Conceptual Overview

```
Iteration 0:  predict mean (F₀)
Iteration 1:  fit a tree to residuals r₁ = y - F₀  →  F₁ = F₀ + η·h₁
Iteration 2:  fit a tree to residuals r₂ = y - F₁  →  F₂ = F₁ + η·h₂
...
Iteration T:  F_T = F₀ + η·Σ hₜ
```

| Concept | Effect |
|---|---|
| **Learning rate η** | Shrinks each tree's contribution — smaller = more trees needed but better generalization |
| **n_estimators** | Number of trees — use early stopping to find optimal T |
| **max_depth** | Tree depth — shallower = more regularization |
| **subsample / colsample** | Stochastic gradient boosting — reduces overfitting |
| **scale_pos_weight** | XGBoost class-imbalance correction = N_neg / N_pos |


In [ ]:
# ── XGBoost with early stopping ───────────────────────────────────────────────
try:
    dtrain = xgb.DMatrix(Xtr,  label=y_tr2, feature_names=feat_nm)
    dval   = xgb.DMatrix(Xval, label=y_val,  feature_names=feat_nm)
    dte    = xgb.DMatrix(Xte,  label=y_te,   feature_names=feat_nm)

    xgb_params = {
        "objective"       : "binary:logistic",
        "eval_metric"     : "auc",
        "eta"             : 0.05,          # learning rate
        "max_depth"       : 6,
        "subsample"       : 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos,     # handle class imbalance
        "seed"            : 42,
        "verbosity"       : 0,
    }
    evals_result = {}
    xgb_model = xgb.train(
        xgb_params, dtrain,
        num_boost_round=500,
        evals=[(dtrain,"train"),(dval,"val")],
        early_stopping_rounds=30,
        evals_result=evals_result,
        verbose_eval=False,
    )
    prob_xgb = xgb_model.predict(dte)
    auc_xgb  = roc_auc_score(y_te, prob_xgb)
    print(f"XGBoost: best_iteration={xgb_model.best_iteration}, AUC={auc_xgb:.4f}")

    # Learning curve
    fig,ax = plt.subplots(figsize=(10,4))
    ax.plot(evals_result["train"]["auc"], color="#4878CF", lw=1.2, alpha=0.8, label="Train AUC")
    ax.plot(evals_result["val"]["auc"],   color="#D65F5F", lw=2,   label="Val AUC")
    ax.axvline(xgb_model.best_iteration, color="green", ls="--", lw=1.5,
               label=f"Best iteration = {xgb_model.best_iteration}")
    ax.set_xlabel("Boosting round"); ax.set_ylabel("AUC")
    ax.set_title("XGBoost learning curve — early stopping")
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.savefig("/tmp/mod04/xgb_learning_curve.png",bbox_inches="tight"); plt.show()
except Exception as e:
    print(f"XGBoost error: {e}"); auc_xgb = 0


In [ ]:
# ── LightGBM — faster, handles categoricals natively ─────────────────────────
try:
    # LightGBM with categorical feature support
    df_lgb = df.copy()
    for col in CATEGORIC:
        df_lgb[col] = df_lgb[col].astype("category")

    X_lgb = df_lgb[NUMERIC+BINARY+CATEGORIC]; y_lgb = df_lgb[TARGET]
    Xl_tr,Xl_te,yl_tr,yl_te=train_test_split(X_lgb,y_lgb,test_size=0.20,stratify=y_lgb,random_state=42)
    Xl_tr2,Xl_val,yl_tr2,yl_val=train_test_split(Xl_tr,yl_tr,test_size=0.15,stratify=yl_tr,random_state=42)

    cat_cols_idx = [X_lgb.columns.get_loc(c) for c in CATEGORIC]
    dtrain_l = lgb.Dataset(Xl_tr2,  label=yl_tr2, categorical_feature=cat_cols_idx)
    dval_l   = lgb.Dataset(Xl_val,  label=yl_val,  categorical_feature=cat_cols_idx, reference=dtrain_l)

    lgb_params = {
        "objective"       : "binary",
        "metric"          : "auc",
        "learning_rate"   : 0.05,
        "num_leaves"      : 31,
        "max_depth"       : -1,
        "subsample"       : 0.8,
        "colsample_bytree": 0.8,
        "is_unbalance"    : True,
        "verbosity"       : -1,
        "seed"            : 42,
    }
    lgb_callbacks = [lgb.early_stopping(stopping_rounds=30, verbose=False),
                     lgb.log_evaluation(period=-1)]
    lgb_evals = {}
    lgb_model = lgb.train(
        lgb_params, dtrain_l,
        num_boost_round=500,
        valid_sets=[dtrain_l, dval_l],
        valid_names=["train","val"],
        callbacks=lgb_callbacks,
    )
    prob_lgb = lgb_model.predict(Xl_te)
    auc_lgb  = roc_auc_score(yl_te, prob_lgb)
    print(f"LightGBM: best_iteration={lgb_model.best_iteration}, AUC={auc_lgb:.4f}")
except Exception as e:
    print(f"LightGBM error: {e}"); auc_lgb = 0


## 3. Optuna Hyperparameter Optimisation

In [ ]:
try:
    from sklearn.ensemble import GradientBoostingClassifier

    def objective(trial):
        params = {
            "n_estimators"  : trial.suggest_int("n_estimators",  100, 500),
            "max_depth"     : trial.suggest_int("max_depth",       3, 8),
            "learning_rate" : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample"     : trial.suggest_float("subsample",     0.6, 1.0),
            "max_features"  : trial.suggest_categorical("max_features", ["sqrt","log2"]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 30),
            "random_state"  : 42,
        }
        model = GradientBoostingClassifier(**params)
        skf   = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = cross_val_score(model, Xtr, y_tr2, cv=skf, scoring="roc_auc")
        return scores.mean()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=30, show_progress_bar=False)

    print(f"Optuna best AUC: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

    # Plot optimisation history
    fig, axes = plt.subplots(1,2,figsize=(13,4))
    trial_vals = [t.value for t in study.trials]
    best_so_far = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]
    axes[0].plot(trial_vals,    "o", color="#AEC6CF", ms=5, alpha=0.6, label="Trial AUC")
    axes[0].plot(best_so_far,   "-", color="#1F4E79", lw=2, label="Best so far")
    axes[0].set_xlabel("Trial"); axes[0].set_ylabel("CV AUC")
    axes[0].set_title("Optuna optimisation history"); axes[0].legend(fontsize=9)

    # Param importance
    param_imp = optuna.importance.get_param_importances(study)
    axes[1].barh(list(param_imp.keys())[::-1], list(param_imp.values())[::-1],
                  color="#D65F5F", alpha=0.85)
    axes[1].set_xlabel("Importance"); axes[1].set_title("Optuna parameter importance")
    plt.tight_layout(); plt.savefig("/tmp/mod04/optuna.png",bbox_inches="tight"); plt.show()

except Exception as e:
    print(f"Optuna not available or error: {e}")
    print("Install with: pip install optuna")


## 4. Model Comparison Summary

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

lr=LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42)
lr.fit(Xtr,y_tr2); p_lr=lr.predict_proba(Xte)[:,1]
rf=RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced",random_state=42,n_jobs=-1)
rf.fit(Xtr,y_tr2); p_rf=rf.predict_proba(Xte)[:,1]

model_probs = {"Logistic Reg":p_lr,"Random Forest":p_rf}
if auc_xgb>0: model_probs["XGBoost"]=prob_xgb
if auc_lgb>0: model_probs["LightGBM"]=prob_lgb

fig, axes = plt.subplots(1,2,figsize=(14,5))
colors_m={"Logistic Reg":"#6ACC65","Random Forest":"#4878CF","XGBoost":"#D65F5F","LightGBM":"#B47CC7"}

summary = []
for name,prob in model_probs.items():
    auc  = roc_auc_score(y_te,prob)
    ap   = average_precision_score(y_te,prob)
    brier= brier_score_loss(y_te,prob)
    summary.append({"Model":name,"AUC":round(auc,4),"Avg Precision":round(ap,4),"Brier":round(brier,4)})
    fpr,tpr,_=roc_curve(y_te,prob)
    axes[0].plot(fpr,tpr,lw=2,color=colors_m.get(name,"gray"),label=f"{name} ({auc:.3f})")

axes[0].plot([0,1],[0,1],"k--",lw=1); axes[0].set_xlabel("1-Specificity"); axes[0].set_ylabel("Sensitivity")
axes[0].set_title("ROC curves — Gradient Boosting comparison"); axes[0].legend(fontsize=9,loc="lower right")

summ_df=pd.DataFrame(summary).sort_values("AUC",ascending=False)
x=np.arange(len(summ_df)); w=0.25
axes[1].bar(x-w,   summ_df["AUC"],          w,label="AUC",color="#4878CF",alpha=0.85)
axes[1].bar(x,     summ_df["Avg Precision"],w,label="Avg Precision",color="#D65F5F",alpha=0.85)
axes[1].bar(x+w,   1-summ_df["Brier"],      w,label="1-Brier",color="#6ACC65",alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(summ_df["Model"],rotation=15)
axes[1].set_ylabel("Score"); axes[1].set_title("Model comparison — key metrics")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod04/xgb_lgb_comparison.png",bbox_inches="tight"); plt.show()
print("\nModel comparison table:")
display(summ_df)


## 5. Knowledge Check
1. XGBoost stopped at iteration 87 out of 500. What does this tell you about the learning rate and regularization?
2. LightGBM grows leaf-wise while XGBoost grows level-wise. What are the practical implications for clinical datasets with few features?
3. `scale_pos_weight` was set to 6.8 (ratio of negatives to positives). What would happen if you forgot this parameter?
4. Your Optuna study ran 30 trials. The best AUC is 0.78. Would 100 more trials likely improve this substantially?
5. Re-run LightGBM with `learning_rate=0.01` and `num_boost_round=1000`. Does it converge to the same AUC?


In [ ]:
# Exercise 5 — LightGBM with lower learning rate
try:
    lgb_params_slow = {**lgb_params, "learning_rate":0.01}
    lgb_slow = lgb.train(
        lgb_params_slow, dtrain_l, num_boost_round=1000,
        valid_sets=[dval_l], callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)]
    )
    prob_slow = lgb_slow.predict(Xl_te)
    print(f"LightGBM lr=0.05: AUC={auc_lgb:.4f}  (best_iter={lgb_model.best_iteration})")
    print(f"LightGBM lr=0.01: AUC={roc_auc_score(yl_te,prob_slow):.4f}  (best_iter={lgb_slow.best_iteration})")
    print("Smaller LR needs more iterations but often achieves similar final AUC.")
except Exception as e:
    print(f"Error: {e}")


***
**Next → NB-04: Clinical Model Evaluation**
